## Calculate scores from predictions files

In [1]:
import os
import sys

import numpy as np
import pandas as pd
import xarray as xr
import colorcet as cc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

base_dir = os.path.dirname(os.path.abspath('')).split(os.sep + 'eval_transferability')[0]

sys.path.append(os.path.sep.join([base_dir , 'src']))
from help_fcts import get_rmse, get_mae, get_mbe, get_r2, calc_anomaly

In [ ]:

MODEL_SPEC = 'CESM2future'
DATA_SPEC = 'ECEfuture'

if 'future' in DATA_SPEC:
    DD = DATA_SPEC.replace('future','')
    ds_clim = xr.open_dataset(os.path.sep.join([base_dir, 'data', 'interim', DD, 'HIRHAM5', 'firnpack', 'climat_2075-2100_smoothed15.nc']))
else: 
    ds_clim = xr.open_dataset(os.path.sep.join([base_dir, 'data', 'interim', DATA_SPEC, 'HIRHAM5', 'firnpack', 'climat_1990-2016_smoothed15.nc']))
snmel_clim = ds_clim['snmel']

pred_dir = os.path.sep.join([base_dir, 'output', f'repeated_{MODEL_SPEC}_modularNNEBMphysical20onlyabove10mm_log'])
    
chunking = {'time': 366, 'y': -1, 'x': -1}

In [3]:
# # For mixed training

# MODEL_SPEC = 'ERAICESM2'
# DATA_SPEC = 'ERAI'

# ds_clim = xr.open_dataset(os.path.sep.join([base_dir, 'data', 'interim', DATA_SPEC, 'HIRHAM5', 'firnpack', 'climat_1990-2016_smoothed15.nc']))
# snmel_clim = ds_clim['snmel']

# pred_dir = os.path.sep.join([base_dir, 'output', f'{MODEL_SPEC}_2500_modularNNEBM_log'])

# # fig_dir = os.path.sep.join([base_dir, 'figures_crossval', f'modularEBM'])
# # os.makedirs(fig_dir, exist_ok=True)

# if MODEL_SPEC == DATA_SPEC:
#     FILE_SPEC = ''
# else:
#     FILE_SPEC = '_' + DATA_SPEC
    
# chunking = {'time': 366, 'y': -1, 'x': -1}

## Load data

In [ ]:
# Calculate scores for all years

# if cross-validation
valyears = range(1992, 2017, 2)   

# if repeated training; years is placeholder and does not have a function
valyears = [2016]  

# years to calculate scores for:
if 'future' in DATA_SPEC:
    appliedyears = range(2075, 2101)
else:
    appliedyears = range(1990, 2017)

iter_n = range(20)
scaling_factor = 1.796553703424 / 58391 

metrics = ['true melt', 'predicted melt', 'residual', 'residual_rel',
            'RMSE', 'RMSEn', 'MAE', 'MAEn', 'MBE', 'MBEn', 'R2', 'R2_anomalies', 
            'residual_leq1', 'residual_leq50', 'residual_g50', 'RMSE_debiased']

# Create multi-level index (years × categories)
index = pd.MultiIndex.from_product([valyears, iter_n, appliedyears], names=['valyear', 'iter', 'appliedyear'])

# # Create DataFrame with this index and metric columns
df_per_year = pd.DataFrame(np.nan,  index=index, dtype=float, columns=metrics)
df_jja_per_year = pd.DataFrame(np.nan,  index=index, dtype=float, columns=metrics)


# data_dir = os.path.sep.join([pred_dir, f'fold_{valyears[0]}', f'pred_data_{DATA_SPEC}.zarr']) # if cross-validation with multiple years
data_dir = os.path.sep.join([pred_dir, f'pred_data_{DATA_SPEC}.zarr'])
with xr.open_zarr(data_dir).chunk(chunking) as ds:
    da_clim = ds_clim['snmel'].sel(x=ds['x'].astype(int), y=ds['y'].astype(int))

true_da_cache = {}
mask_year_cache = {}
mask_summer_cache = {}
true_da_jja_cache = {}
total_melt_cache = {}
total_melt_jja_cache = {}
mean_melt_cache ={}
mean_melt_jja_cache = {}

for year in valyears:
    # data_dir = os.path.sep.join([pred_dir, f'fold_{year}', f'pred_data_{DATA_SPEC}.zarr'])  # if cross-validation with multiple years
    data_dir = os.path.sep.join([pred_dir, f'pred_data_{DATA_SPEC}.zarr'])

    print(f'Load data {data_dir} ...')
    with xr.open_zarr(data_dir).chunk(chunking) as ds:
        ds_true = ds['snmel_true'].persist()
        ds_pred_full = ds['snmel_pred']

        for it in iter_n:
            try:
                print(f'Calculate scores for iteration nr {it} ...')
                pred_it = ds_pred_full.sel(iteration=it).persist()

                for ay in appliedyears:
                    print(f'Calculate scores for year {ay}...')

                    key = (year, ay)
                    if key not in true_da_cache:

                        mask_year = slice(f"{ay}-01-01", f"{ay}-12-31")
                        true_da = ds_true.sel(time=mask_year).compute()

                        mask_summer = true_da.time.dt.month.isin([6, 7, 8])
                        true_da_jja = true_da.sel(time=mask_summer).compute()

                        total_melt = int(round(true_da.sum(dim=['time','x','y']).item()*scaling_factor,0))
                        total_melt_jja = int(round(true_da_jja.sum(dim=['time','x','y']).item()*scaling_factor,0))
                        mean_melt = true_da.mean(dim=['time','x','y']).item()*scaling_factor
                        mean_melt_jja = true_da_jja.mean(dim=['time','x','y']).item()*scaling_factor

                        # caching
                        mask_year_cache[key] = mask_year
                        true_da_cache[key] = true_da
                        mask_summer_cache[key] = mask_summer
                        true_da_jja_cache[key] = true_da_jja
                        total_melt_cache[key] = total_melt
                        total_melt_jja_cache[key] = total_melt_jja
                        mean_melt_cache[key] = mean_melt
                        mean_melt_jja_cache[key] = mean_melt_jja
                    else:
                        mask_year = mask_year_cache[key]
                        true_da = true_da_cache[key]
                        mask_summer = mask_summer_cache[key]
                        true_da_jja = true_da_jja_cache[key]
                        total_melt = total_melt_cache[key]
                        total_melt_jja = total_melt_jja_cache[key]
                        mean_melt = mean_melt_cache[key]
                        mean_melt_jja = mean_melt_jja_cache[key]


                    
                    pred_da = pred_it.sel(time=mask_year).compute()   
                    pred_da_jja = pred_da.sel(time=mask_summer).compute()

                    residual = (pred_da - true_da)
                    residual_melt_g50 = residual.where(true_da > 50)
                    residual_melt_leq50 = residual.where((1 < true_da) & (true_da <= 50))
                    residual_melt_leq1 = residual.where(true_da <= 1)
                                       
                    # statistical scores whole year
                    df_per_year.loc[(year,it,ay), 'true melt'] = total_melt
                    df_per_year.loc[(year,it,ay), 'predicted melt'] = int(round(pred_da.sum(dim=['time','x','y']).item()*scaling_factor,0))
                    
                    df_per_year.loc[(year,it,ay), 'residual'] = residual.sum(dim=['time','x','y']).item()*scaling_factor
                    df_per_year.loc[(year,it,ay), 'residual_g50'] = residual_melt_g50.sum(dim=['time','x','y']).item()*scaling_factor
                    df_per_year.loc[(year,it,ay), 'residual_leq50'] = residual_melt_leq50.sum(dim=['time','x','y']).item()*scaling_factor
                    df_per_year.loc[(year,it,ay), 'residual_leq1'] = residual_melt_leq1.sum(dim=['time','x','y']).item()*scaling_factor
                    df_per_year.loc[(year,it,ay), 'residual_rel'] = df_per_year.loc[(year,it,ay), 'residual'] / total_melt
                    
                    df_per_year.loc[(year,it,ay), 'RMSE'] = get_rmse(residual)
                    df_per_year.loc[(year,it,ay), 'RMSEn'] = df_per_year.loc[(year,it,ay), 'RMSE'] / mean_melt
                    df_per_year.loc[(year,it,ay), 'MAE'] = get_mae(residual)
                    df_per_year.loc[(year,it,ay), 'MAEn'] = df_per_year.loc[(year,it,ay), 'MAE'] / mean_melt
                    df_per_year.loc[(year,it,ay), 'MBE'] = get_mbe(residual)
                    df_per_year.loc[(year,it,ay), 'MBEn'] = df_per_year.loc[(year,it,ay), 'MBE'] / mean_melt
                    df_per_year.loc[(year,it,ay), 'R2'] = get_r2(true_da.values, pred_da.values)

                    residual_debiased = residual - df_per_year.loc[(year,it,ay), 'MBE']
                    df_per_year.loc[(year,it,ay), 'RMSE_debiased'] = get_rmse(residual_debiased)

                    anomaly_pred, _, _ = calc_anomaly(pred_da, da_clim)
                    anomaly_true, _, _ = calc_anomaly(true_da, da_clim)
                    df_per_year.loc[(year,it,ay), 'R2_anomalies'] = get_r2(anomaly_true.values, anomaly_pred.values)
                    


                    # statistical scores for JJA
                    residual_jja = (pred_da_jja - true_da_jja)
                    residual_jja_melt_g50 = residual_jja.where(true_da_jja > 50)
                    residual_jja_melt_leq50 = residual_jja.where((true_da_jja > 1) & (true_da_jja <= 50))
                    residual_jja_melt_leq1 = residual_jja.where(true_da_jja <= 1)
                    
                    df_jja_per_year.loc[(year,it,ay), 'true melt'] = total_melt_jja
                    df_jja_per_year.loc[(year,it,ay), 'predicted melt'] = int(round(pred_da_jja.sum(dim=['time','x','y']).compute().item()*scaling_factor,0))
                
                    df_jja_per_year.loc[(year,it,ay), 'residual'] = residual.sum(dim=['time','x','y']).item()*scaling_factor
                    df_jja_per_year.loc[(year,it,ay), 'residual_g50'] = residual_jja_melt_g50.sum(dim=['time','x','y']).item()*scaling_factor
                    df_jja_per_year.loc[(year,it,ay), 'residual_leq50'] = residual_jja_melt_leq50.sum(dim=['time','x','y']).item()*scaling_factor
                    df_jja_per_year.loc[(year,it,ay), 'residual_leq1'] = residual_jja_melt_leq1.sum(dim=['time','x','y']).item()*scaling_factor
                    df_jja_per_year.loc[(year,it,ay), 'residual_rel'] = df_jja_per_year.loc[(year,it,ay), 'residual'] / total_melt_jja
                    
                    df_jja_per_year.loc[(year,it,ay), 'RMSE'] = get_rmse(residual.where(true_da_jja.notnull()))
                    df_jja_per_year.loc[(year,it,ay), 'RMSEn'] = df_jja_per_year.loc[(year,it,ay), 'RMSE'] / mean_melt_jja
                    df_jja_per_year.loc[(year,it,ay), 'MAE'] = get_mae(residual.where(true_da_jja.notnull()))
                    df_jja_per_year.loc[(year,it,ay), 'MAEn'] = df_jja_per_year.loc[(year,it,ay), 'MAE'] / mean_melt_jja
                    df_jja_per_year.loc[(year,it,ay), 'MBE'] = get_mbe(residual.where(true_da_jja.notnull()))
                    df_jja_per_year.loc[(year,it,ay), 'MBEn'] = df_jja_per_year.loc[(year,it,ay), 'MBE'] / mean_melt_jja
                    pred_da_JJA = pred_da.sel(time=pred_da.time.dt.month.isin([6, 7, 8]))
                    df_jja_per_year.loc[(year,it,ay), 'R2'] = get_r2(true_da_jja.values, pred_da_JJA.values)

                    residual_debiased_jja = residual.where(true_da_jja.notnull()) - df_jja_per_year.loc[(year,it,ay), 'MBE']
                    df_jja_per_year.loc[(year,it,ay), 'RMSE_debiased'] = get_rmse(residual_debiased_jja)

                    anomaly_pred_jja = anomaly_pred.sel(time=anomaly_pred.time.dt.month.isin([6, 7, 8]))
                    anomaly_true_jja = anomaly_true.sel(time=anomaly_true.time.dt.month.isin([6, 7, 8]))
                    df_jja_per_year.loc[(year,it,ay), 'R2_anomalies'] = get_r2(anomaly_true_jja.values, anomaly_pred_jja.values)

            except Exception as e:
                print(e)
            # Write after each year
            output_file = os.path.sep.join([pred_dir, f'scores_per_year_{DATA_SPEC}.csv'])
            df_per_year.to_csv(output_file)
            print(f'Scores saved to {output_file} (after predicting year {ay})')

            output_file_JJA = os.path.sep.join([pred_dir, f'scores_per_year_JJA_{DATA_SPEC}.csv'])
            df_jja_per_year.to_csv(output_file_JJA)
            print(f'Scores saved to {output_file_JJA} (after predicting year {ay})')

display(df_per_year.round(2))

Load data /home/eschlager/MeltEmulation/output/repeated_CESM2future_modularNNEBMphysical20onlyabove10mm_log/pred_data_ECEfuture.zarr ...


Calculate scores for iteration nr 0 ...


Calculate scores for year 2075...


Calculate scores for year 2076...


Calculate scores for year 2077...


...

Calculate scores for year 2098...


Calculate scores for year 2099...


Calculate scores for year 2100...


Scores saved to /home/eschlager/MeltEmulation/output/repeated_CESM2future_modularNNEBMphysical20onlyabove10mm_log/scores_per_year_ECEfuture.csv (after predicting year 2100)
Scores saved to /home/eschlager/MeltEmulation/output/repeated_CESM2future_modularNNEBMphysical20onlyabove10mm_log/scores_per_year_JJA_ECEfuture.csv (after predicting year 2100)


true melt  predicted melt  residual  residual_rel  \
valyear iter appliedyear                                                      
2016    0    2075            1899.0          2108.0    208.61          0.11   
             2076            2266.0          2400.0    134.50          0.06   
             2077            2350.0          2586.0    235.51          0.10   
             2078            2212.0          2394.0    181.67          0.08   
             2079            2536.0          2722.0    185.29          0.07   
...                             ...             ...       ...           ...   
        19   2096            2522.0          2717.0    195.03          0.08   
             2097            3201.0          3422.0    221.01          0.07   
             2098            3706.0          3873.0    167.16          0.05   
             2099            3147.0          3305.0    158.78          0.05   
             2100            2787.0          3078.0    291.14          0.10   

                          RMSE     RMSEn   MAE     MAEn   MBE     MBEn    R2  \
valyear iter appliedyear                                                       
2016    0    2075         1.91  21429.18  0.69  7726.34  0.32  3569.87  0.95   
             2076         1.58  14937.10  0.58  5488.52  0.20  1929.42  0.97   
             2077         1.83  16595.23  0.68  6157.07  0.36  3256.83  0.97   
             2078         1.73  16681.25  0.66  6368.05  0.28  2669.05  0.97   
             2079         1.85  15529.75  0.70  5869.94  0.28  2374.44  0.97   
...                        ...       ...   ...      ...   ...      ...   ...   
        19   2096         1.83  15485.63  0.65  5484.03  0.30  2513.72  0.96   
             2097         1.97  13143.79  0.71  4711.72  0.34  2244.40  0.97   
             2098         1.94  11179.79  0.69  3994.73  0.25  1465.93  0.98   
             2099         2.06  13952.78  0.75  5096.24  0.24  1640.10  0.97   
             2100         2.07  15861.81  0.73  5577.49  0.44  3395.42  0.96   

                          R2_anomalies  residual_leq1  residual_leq50  \
valyear iter appliedyear                                                
2016    0    2075                 0.79         219.36          -12.20   
             2076                 0.88         145.18          -11.58   
             2077                 0.84         214.08           17.28   
             2078                 0.86         191.94          -10.46   
             2079                 0.83         184.58           -1.17   
...                                ...            ...             ...   
        19   2096                 0.80         127.30           63.61   
             2097                 0.87         101.88          108.75   
             2098                 0.91          73.33           83.29   
             2099                 0.84          91.21           62.02   
             2100                 0.80         148.06          134.20   

                          residual_g50  RMSE_debiased  
valyear iter appliedyear                               
2016    0    2075                 1.45           1.88  
             2076                 0.90           1.57  
             2077                 4.14           1.79  
             2078                 0.19           1.71  
             2079                 1.88           1.83  
...                                ...            ...  
        19   2096                 4.13           1.80  
             2097                10.38           1.94  
             2098                10.54           1.93  
             2099                 5.55           2.05  
             2100                 8.88           2.03  

[520 rows x 16 columns]

In [ ]:
if DATA_SPEC == MODEL_SPEC:

    # --------------- FOR CROSSVALIDATION, BEST ITERATION WITH RESPECT TO VALIDATION YEAR ----------------- #
    filtered_df = df_per_year[df_per_year.index.get_level_values(0) == df_per_year.index.get_level_values(2)]
    dff = filtered_df.loc[
        filtered_df.groupby(level=0)['MBE'].apply(
            lambda x: (x.abs()).idxmin()
        )
    ]
    # for each appliedyear, mark the model which had lowest MBE on its validation year
    df_per_year['best_mbe'] = df_per_year.index.droplevel(2).isin(dff.index.droplevel(2))


    filtered_df = df_per_year[df_per_year.index.get_level_values(0) == df_per_year.index.get_level_values(2)]
    dff = filtered_df.loc[
        filtered_df.groupby(level=0)['MAE'].apply(
            lambda x: x.idxmin()
        )
    ]
    # for each appliedyear, mark the model which had lowest MAE on its validation year
    df_per_year['best_mae'] = df_per_year.index.droplevel(2).isin(dff.index.droplevel(2))


    filtered_df = df_per_year[df_per_year.index.get_level_values(0) == df_per_year.index.get_level_values(2)]
    dff = filtered_df.loc[
        filtered_df.groupby(level=0)['RMSE'].apply(
            lambda x: x.idxmin()
        )
    ]
    # for each appliedyear, mark the model which had lowest RMSE on its validation year
    df_per_year['best_rmse'] = df_per_year.index.droplevel(2).isin(dff.index.droplevel(2))


    # --------------- FOR REPEATED TRAINING, BEST ITERATION WITH RESPECT TO ALL YEARS ----------------- #
    # calculate lowest RMSE across all years
    filtered_df = df_per_year.groupby(['valyear', 'iter']).mean()
    dff = filtered_df.loc[filtered_df.groupby('valyear')['RMSE'].idxmin()]
    df_per_year['best_rmse_allyears'] = df_per_year.index.droplevel(2).isin(dff.index)

    # calculate lowest MAE across all years
    filtered_df = df_per_year.groupby(['valyear', 'iter']).mean()
    dff = filtered_df.loc[filtered_df.groupby('valyear')['MAE'].idxmin()]
    df_per_year['best_mae_allyears'] = df_per_year.index.droplevel(2).isin(dff.index)

    # calculate lowest MBE across all years
    filtered_df = df_per_year.abs().groupby(['valyear', 'iter']).mean()
    dff = filtered_df.loc[filtered_df.groupby('valyear')['MBE'].idxmin()]
    df_per_year['best_mbe_allyears'] = df_per_year.index.droplevel(2).isin(dff.index)


    # ----------------- overwrite scores file ------------------- #
    output_file = os.path.sep.join([pred_dir, f'scores_per_year_{DATA_SPEC}.csv'])
    df_per_year.to_csv(output_file)